# 03 — Thermal Power Analysis

Read `outputs/thermal_power_table.csv` (written by Stage 11) and produce
an extended thermal power analysis: cumulative energy, capacity factor, and
comparison with the undisturbed reference.

**Prerequisites**: Stage 11 must have completed.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'scripts'))

from config import load_config, OUTPUTS_DIR

cfg = load_config()

csv_path = OUTPUTS_DIR / 'thermal_power_table.csv'
df = pd.read_csv(csv_path)
print(df.head())

In [ ]:
# Reference initial power (undisturbed reservoir)
_RHO_CP = 4.1868e6   # J/(m3·K)
Q_m3s   = 5 * 30e-3  # 5 wells × 30 L/s
P0_MW   = _RHO_CP * Q_m3s * (cfg.slice_T[1] - cfg.T_inj) / 1e6

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df['time_yr'], df['P_MW'], 'b-o', markersize=4, lw=1.5,
        label='Simulated P_th')
ax.axhline(P0_MW, color='gray', linestyle='--',
           label=f'P0 = {P0_MW:.2f} MW_th (undisturbed)')
ax.fill_between(df['time_yr'], df['P_MW'], P0_MW,
                alpha=0.1, color='red', label='Power loss due to breakthrough')
ax.set_xlabel('Time [yr]')
ax.set_ylabel('Thermal power [MW_th]')
ax.set_title('Group 3 — thermal power over 100 years')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)
ax.set_ylim(0)
plt.tight_layout()
plt.show()

In [ ]:
# Cumulative thermal energy [GWh]
# Integrate P_th [MW] over time [yr] using the trapezoidal rule
E_GWh = np.trapz(df['P_MW'], df['time_yr']) * 8760 / 1000  # MWh → GWh

print(f'Cumulative thermal energy (100 yr): {E_GWh:.1f} GWh')
print(f'Average thermal power             : {E_GWh * 1000 / (100 * 8760):.2f} MW_th')
print(f'Capacity factor at 100 yr         : {df["P_MW"].iloc[-1] / P0_MW * 100:.1f} %')